# Fiducial pair reduction
The circuits used in standard Long Sequence GST are more than what are needed to amplify every possible gate error.  (Technically, this is due to the fact that the informationally complete fiducial sub-sequences allow extraction of each germ's *entire* process matrix, when all that is needed is the part describing the amplified directions in model space.) Because of this over-completeness, fewer sequences, i.e. experiments, may be used whilst retaining the desired Heisenberg-like scaling ($\sim 1/L$, where $L$ is the maximum length sequence).  The over-completeness can still be desirable, however, as it makes the GST optimization more robust to model violation and so can serve to stabilize the GST parameter optimization in the presence of significant non-Markovian noise.  Recall that the form of a GST gate sequence is

$$S = F_i (g_k)^n F_j $$

where $F_i$ is a "preparation fiducial" sequence, $F_j$ is a "measurement fiducial" sequence, and $g_k$ is a "germ" sequence.  The repeated germ sequence $(g_k)^n$ we refer to as a "germ-power".

## The `fpr=True` shortcut

For a model pack, fiducial pair reduction is a single argument: `create_gst_experiment_design` accepts `fpr=True` and applies per-germ fiducial pair sets precomputed for the pack.

In [1]:
from pygsti.modelpacks import smq1Q_XY

full_design = smq1Q_XY.create_gst_experiment_design(max_max_length=32)
fpr_design = smq1Q_XY.create_gst_experiment_design(max_max_length=32, fpr=True)
print("Without FPR: %d circuits" % len(full_design.all_circuits_needing_data))
print("With fpr=True: %d circuits" % len(fpr_design.all_circuits_needing_data))

Without FPR: 568 circuits
With fpr=True: 128 circuits


For this pack, 128 circuits instead of 568.  The rest of this page covers the algorithms behind that argument: what to reach for when no model pack covers your gate set, or when you want control over how the pairs are chosen.  To check that a reduced design is still adequate, compute its Fisher information ([evaluating experiment designs](CheckYourDesign)); to estimate what a design costs to run, use `calculate_edesign_estimated_runtime` ([experiment designs](../workflow/ExperimentDesigns)).

## The four reduction methods

There are currently four different ways to reduce a standard set of GST operation sequences within pyGSTi, each of which removes certain $(F_i,F_j)$ fiducial pairs for certain germ-powers.

- **Per-germ fiducial pair reduction (PFPR)** removes the same intelligently-selected set of fiducial pairs for all powers of a given germ, but different sets are removed for different germs.  Since different germs amplify different directions in model space, it makes intuitive sense to specify different fiducial pair sets for different germs.  Because this method only considers one germ at a time, its search stays cheap as the gate set grows.

- **Per-germ global fiducial pair reduction (PGGFPR)** removes the same intelligently-selected set of fiducial pairs for all powers of a given germ, but different sets are removed for different germs while also taking into account the amplificational properties of a germ set as a whole. This is a two-step process in which we first identify redundancy within a germ set itself due to overlapping amplified directions in parameter space and identifies a subset of amplified parameters for each germ such that collectively we have sensitivity to every direction. In the second stage we select a subset of fiducial pairs for each germ only requiring sensitivity to the subset of amplified parameters of that germ identified in the first stage. With the right settings this method can achieve experiment designs approaching information-theoretic lower bounds in size — savings that materialize on multi-qubit design problems rather than in the single-qubit demo below. With fewer fiducial pairs comes the potential for detecting non-Markovian effects and potentially less robustness to those effects (the extent to which this is true, or if it is true at all, is an active area of research), so caveat emptor.

- **Random per-germ power fiducial pair reduction (RFPR)** randomly chooses a different set of fiducial pairs to remove for each germ-power.  It is extremely fast to perform, as pairs are just randomly selected for removal, and in practice works well (i.e. does not impair Heisenberg-scaling) up until some critical fraction of the pairs are removed.  This reflects the fact that the direction detected by a fiducial pair usually has some non-negligible overlap with each of the directions amplified by a germ, and it is the exceptional case that an amplified direction escapes undetected.  As such, the "critical fraction" which can usually be safely removed equals the ratio of amplified-parameters to germ-process-matrix-elements (typically $\approx 1/d^2$ where $d$ is the Hilbert space dimension, so $1/4 = 25\%$ for 1 qubit and $1/16 = 6.25\%$ for 2 qubits).  RFPR can be combined with PFPR so that some number of randomly chosen pairs can be added on top of the "intelligently-chosen" pairs of PFPR.  In this way, one can vary the amount of sequence reduction (in order to trade off speed vs. robustness to non-Markovian noise) without inadvertently selecting too few or an especially bad set of random fiducial pairs.

- **Global fiducial pair reduction (GFPR)** removes a single, intelligently-selected set of fiducial pairs for all germs and all germ-powers, chosen by repeatedly evaluating the number of amplified parameters for an *entire germ set*.  That global search is what rules it out in practice: the number of candidate pair lists grows combinatorially with the number of pairs kept — watch its demo below escalate from 630 candidate lists to test up to 376,992.

```{warning}
GFPR's `find_sufficient_fiducial_pairs` is deprecated in favor of `find_sufficient_fiducial_pairs_per_germ`, and Python's default warning filters usually hide the deprecation warning at runtime. Use a per-germ method in new code; the GFPR section below is kept for reference.
```

## Preliminaries

We now demonstrate how to invoke each of these methods within pyGSTi for the case of a single qubit, using our standard $X(\pi/2)$, $Y(\pi/2)$, $I$ model.  First, we retrieve a target `Model` as usual, along with corresponding sets of fiducial and germ sequences.  We set the maximum length to be 32, roughly consistent with our data-generating model having gates depolarized by 10%.

In [2]:
#Import pyGSTi and the "standard 1-qubit quantities for a model with X(pi/2), Y(pi/2)"
import pygsti
import pygsti.circuits as pc
from pygsti.modelpacks import smq1Q_XY
import numpy as np

#Collect a target model, germ and fiducial strings, and set 
# a list of maximum lengths.
target_model = smq1Q_XY.target_model()
prep_fiducials = smq1Q_XY.prep_fiducials()
meas_fiducials = smq1Q_XY.meas_fiducials()
germs = smq1Q_XY.germs()
maxLengths = [1,2,4,8,16,32]

opLabels = list(target_model.operations.keys())
print("Gate operation labels = ", opLabels)

Gate operation labels =  [Label(('Gxpi2', 0)), Label(('Gypi2', 0))]


## Sequence reduction

Now let's generate a list of all the operation sequences for each maximum length - so a list of lists.  We'll generate the full lists (without any reduction) and the lists for each of the four reduction types listed above.  In the random reduction case, we'll keep 30% of the fiducial pairs, removing 70% of them.  One constant to expect in every table that follows: the L=1 entry is always the same 56 sequences.  That first list is just the LGST circuits, which are always kept — reduction only thins the germ-power circuits stacked on top of them.

### No reduction ("standard" GST)

In [3]:
#Make list-of-lists of GST operation sequences
fullStructs = pc.create_lsgst_circuit_lists(
    opLabels, prep_fiducials, meas_fiducials, germs, maxLengths)

#Print the number of operation sequences for each maximum length
print("** Without any reduction ** ")
for L,strct in zip(maxLengths,fullStructs):
    print("L=%d: %d operation sequences" % (L,len(strct)))
    
#Make a (single) list of all the GST sequences ever needed,
# that is, the list of all the experiments needed to perform GST.
fullExperiments = pc.create_lsgst_circuits(
    opLabels, prep_fiducials, meas_fiducials, germs, maxLengths)
print("\n%d experiments to run GST." % len(fullExperiments))

** Without any reduction ** 
L=1: 56 operation sequences
L=2: 96 operation sequences
L=4: 177 operation sequences
L=8: 304 operation sequences
L=16: 436 operation sequences
L=32: 568 operation sequences

568 experiments to run GST.


### Per-germ fiducial pair reduction (PFPR)

In [4]:
fid_pairsDict = pygsti.alg.find_sufficient_fiducial_pairs_per_germ(
                target_model, prep_fiducials, meas_fiducials, germs,
                search_mode="random", constrain_to_tp=True,
                n_random=100, min_iterations=50,
                base_loweig_tol= .25, num_soln_returned=1,
                type_soln_returned= 'best',
                retry_for_smaller=True,
                seed=1234, verbosity=1,
                mem_limit=int(2*(1024)**3))
print("\nPer-germ FPR to keep the pairs:")
for germ,pairsToKeep in fid_pairsDict.items():
    print("%s: %s" % (str(germ),pairsToKeep))

pfprStructs = pc.create_lsgst_circuit_lists(
    opLabels, prep_fiducials, meas_fiducials, germs, maxLengths,
    fid_pairs=fid_pairsDict) #note: fid_pairs arg can be a dict too!

print("\nPer-germ FPR reduction")
for L,strct in zip(maxLengths,pfprStructs):
    print("L=%d: %d operation sequences" % (L,len(strct)))

pfprExperiments = pc.create_lsgst_circuits(
    opLabels, prep_fiducials, meas_fiducials, germs, maxLengths,
    fid_pairs=fid_pairsDict)
print("\n%d experiments to run GST." % len(pfprExperiments))

------  Per Germ (L=1) Fiducial Pair Reduction --------
Progress: [##################################################] 100.0% -- Circuit(Gxpi2:0Gxpi2:0Gypi2:0@(0)) germ (5 params)

Per-germ FPR to keep the pairs:
Qubit 0 ---|Gxpi2|---
: [(0, 4), (1, 4), (5, 5)]
Qubit 0 ---|Gypi2|---
: [(0, 0), (0, 5), (1, 1), (4, 4)]
Qubit 0 ---|Gxpi2|-|Gypi2|---
: [(0, 2), (0, 3), (0, 4), (0, 5), (1, 4), (2, 3), (3, 0), (4, 0), (5, 0)]
Qubit 0 ---|Gxpi2|-|Gxpi2|-|Gypi2|---
: [(0, 1), (0, 2), (0, 5), (2, 5), (3, 1), (3, 3), (4, 1), (4, 2), (4, 5)]

Per-germ FPR reduction
L=1: 56 operation sequences
L=2: 60 operation sequences
L=4: 79 operation sequences
L=8: 104 operation sequences
L=16: 129 operation sequences
L=32: 154 operation sequences

154 experiments to run GST.


### Per-germ fiducial pair reduction (PFPR) with greedy search heuristics

In addition to the implementation of per-germ fiducial pair reduction above, which supports either a brute force sequential or random search heuristic, there is also an implementation using a greedy search heuristic combined with fast low-rank update-based techniques for significantly faster execution, particularly when generating experiment designs for two-or-more qubits.

In [5]:
fid_pairsDict = pygsti.alg.find_sufficient_fiducial_pairs_per_germ_greedy(target_model, prep_fiducials, meas_fiducials,
                                                germs, seed=1234, verbosity=1)
print("\nPer-germ FPR to keep the pairs:")
for germ,pairsToKeep in fid_pairsDict.items():
    print("%s: %s" % (str(germ),pairsToKeep))

pfprStructs_greedy = pc.create_lsgst_circuit_lists(
    opLabels, prep_fiducials, meas_fiducials, germs, maxLengths,
    fid_pairs=fid_pairsDict) #note: fid_pairs arg can be a dict too!

print("\nPer-germ FPR reduction (greedy heuristic)")
for L,strct in zip(maxLengths,pfprStructs_greedy):
    print("L=%d: %d operation sequences" % (L,len(strct)))

pfprExperiments_greedy = pc.create_lsgst_circuits(
    opLabels, prep_fiducials, meas_fiducials, germs, maxLengths,
    fid_pairs=fid_pairsDict)
print("\n%d experiments to run GST." % len(pfprExperiments_greedy))

------  Per Germ (L=1) Fiducial Pair Reduction --------
Progress: [##################################################] 100.0% -- Circuit(Gxpi2:0Gxpi2:0Gypi2:0@(0)) germ (5 params)

Per-germ FPR to keep the pairs:
Qubit 0 ---|Gxpi2|---
: [(0, 0), (0, 4), (4, 0), (2, 2)]
Qubit 0 ---|Gypi2|---
: [(0, 0), (0, 5), (4, 1), (1, 1)]
Qubit 0 ---|Gxpi2|-|Gypi2|---
: [(0, 0), (1, 1), (4, 4), (0, 2), (0, 1)]
Qubit 0 ---|Gxpi2|-|Gxpi2|-|Gypi2|---
: [(0, 0), (0, 2), (0, 5), (1, 1), (4, 4), (1, 3), (0, 1)]

Per-germ FPR reduction (greedy heuristic)
L=1: 56 operation sequences
L=2: 60 operation sequences
L=4: 69 operation sequences
L=8: 88 operation sequences
L=16: 107 operation sequences
L=32: 126 operation sequences

126 experiments to run GST.


### Per-germ global fiducial pair reduction (PGGFPR)

As mentioned above, the per-germ global FPR scheme is a two step process. First we identify a reduced set of amplified parameters for each germ to require sensitivity to, and then next we identify reduced sets of fiducials with sensitivity to those particular parameters.

In [6]:
#Note that float_type specifies the numpy data type to use, and is primarily useful
#when needing to fine-tune the memory requirements of the algorithm (running this algorithm for
#more than 2-qubits can be very memory intensive). The correct real or complex typing is automatically inferred
#from the model's basis. When running this function for more than two-qubits, consider
#setting the mode kwarg to 'RRQR', which is typically significantly faster for larger qubit counts, but is slightly
#less performant in terms of the cost function of the returned solutions.
germ_set_spanning_vectors, _ = pygsti.alg.germ_set_spanning_vectors(target_model, germs, float_type= np.double)

#Next use this set of vectors to find a sufficient reduced set of fiducial pairs.
#Alternatively this function can also take as input a list of germs
fid_pairsDict = pygsti.alg.find_sufficient_fiducial_pairs_per_germ_global(target_model, prep_fiducials, meas_fiducials,
                                                germ_vector_spanning_set=germ_set_spanning_vectors, verbosity=1)
print("\nPer-germ Global FPR to keep the pairs:")
for germ,pairsToKeep in fid_pairsDict.items():
    print("%s: %s" % (str(germ),pairsToKeep))

pggfprStructs = pc.create_lsgst_circuit_lists(
    opLabels, prep_fiducials, meas_fiducials, germs, maxLengths,
    fid_pairs=fid_pairsDict) #note: fid_pairs arg can be a dict too!

print("\nPer-germ Global FPR reduction")
for L,strct in zip(maxLengths,pggfprStructs):
    print("L=%d: %d operation sequences" % (L,len(strct)))

pggfprExperiments = pc.create_lsgst_circuits(
    opLabels, prep_fiducials, meas_fiducials, germs, maxLengths,
    fid_pairs=fid_pairsDict)
print("\n%d experiments to run GST." % len(pggfprExperiments))

Number of gauge parameters: 14
Number of non-gauge parameters: 18
Generating compact EVD Cache
Progress: [#########################] 100.0% 
Complete germ set (overcomplete) number of amplified parameters: 26
Returning best found vector set. Final Score: Score: major=-18 minor=40.87297269098422, N: 18
------  Per Germ Global Fiducial Pair Reduction --------
  INVALID LEVEL: Generating compact EVD Cache

  INVALID LEVEL: Generating compact EVD Cache

  INVALID LEVEL: Generating compact EVD Cache

  INVALID LEVEL: Generating compact EVD Cache


Per-germ Global FPR to keep the pairs:
Qubit 0 ---|Gxpi2|---
: [(5, 5), (4, 4), (3, 4), (4, 5), (2, 2), (2, 4)]
Qubit 0 ---|Gypi2|---
: [(5, 5), (1, 1), (3, 5), (5, 1), (4, 4), (1, 5)]
Qubit 0 ---|Gxpi2|-|Gypi2|---
: [(5, 5), (1, 3), (4, 5), (2, 3)]
Qubit 0 ---|Gxpi2|-|Gxpi2|-|Gypi2|---
: [(5, 5), (2, 1)]

Per-germ Global FPR reduction
L=1: 56 operation sequences
L=2: 70 operation sequences
L=4: 88 operation sequences
L=8: 106 operation sequences


### Random fiducial pair reduction (RFPR)

In [7]:
#keep only 30% of the pairs
rfprStructs = pc.create_lsgst_circuit_lists(
    opLabels, prep_fiducials, meas_fiducials, germs, maxLengths,
    keep_fraction=0.30, keep_seed=1234)

print("Random FPR reduction")
for L,strct in zip(maxLengths,rfprStructs):
    print("L=%d: %d operation sequences" % (L,len(strct)))
    
rfprExperiments = pc.create_lsgst_circuits(
    opLabels, prep_fiducials, meas_fiducials, germs, maxLengths,
    keep_fraction=0.30, keep_seed=1234)
print("\n%d experiments to run GST." % len(rfprExperiments))

Random FPR reduction
L=1: 56 operation sequences
L=2: 70 operation sequences
L=4: 97 operation sequences
L=8: 137 operation sequences
L=16: 180 operation sequences
L=32: 222 operation sequences

222 experiments to run GST.


### Global fiducial pair reduction (GFPR)

For reference, the deprecated global method (see the warning above).

In [8]:
fid_pairs = pygsti.alg.find_sufficient_fiducial_pairs(
            target_model, prep_fiducials, meas_fiducials, germs,
            search_mode="random", n_random=10, seed=1234,
            verbosity=1, mem_limit=int(2*(1024)**3), minimum_pairs=2)

# fid_pairs is a list of (prepIndex,measIndex) 2-tuples, where
# prepIndex indexes prep_fiducials and measIndex indexes meas_fiducials
print("Global FPR says we only need to keep the %d pairs:\n %s\n"
      % (len(fid_pairs),fid_pairs))

gfprStructs = pc.create_lsgst_circuit_lists(
    opLabels, prep_fiducials, meas_fiducials, germs, maxLengths,
    fid_pairs=fid_pairs)

print("Global FPR reduction")
for L,strct in zip(maxLengths,gfprStructs):
    print("L=%d: %d operation sequences" % (L,len(strct)))
    
gfprExperiments = pc.create_lsgst_circuits(
    opLabels, prep_fiducials, meas_fiducials, germs, maxLengths,
    fid_pairs=fid_pairs)
print("\n%d experiments to run GST." % len(gfprExperiments))

------  Fiducial Pair Reduction --------
maximum number of amplified parameters = 18
Beginning search for a good set of 2 pairs (630 pair lists to test)
Beginning search for a good set of 3 pairs (7140 pair lists to test)
Beginning search for a good set of 4 pairs (58905 pair lists to test)
Beginning search for a good set of 5 pairs (376992 pair lists to test)
Global FPR says we only need to keep the 5 pairs:
 [(0, 1), (0, 5), (1, 0), (2, 3), (3, 4)]

Global FPR reduction
L=1: 56 operation sequences
L=2: 58 operation sequences
L=4: 69 operation sequences
L=8: 87 operation sequences
L=16: 106 operation sequences
L=32: 125 operation sequences

125 experiments to run GST.


## Running GST
In each case above, we constructed (1) a list-of-lists giving the GST operation sequences for each maximum-length stage, and (2) a list of the experiments.  In what follows, we'll use the experiment list to generate some simulated ("fake") data, and then run GST on it.  Two fits are enough to make the comparison: the full design, and the greedy per-germ design.  Since both are run in exactly the same way, we'll put all of the logic in a function.

We already built each reduced circuit-structure list by hand above, so there's no need to have GST regenerate it: a `GateSetTomographyDesign` accepts a list of circuit structures directly, where a `StandardGSTDesign` would derive them from fiducials, germs, and maximum lengths (the [experiment designs tutorial](../workflow/ExperimentDesigns) covers the distinction). We pair that experiment design with the simulated dataset in a `ProtocolData` object, and hand the pair to the `GateSetTomography` protocol. This sidesteps the old two-driver split between `run_long_sequence_gst`, which always builds a *complete* list of operation sequences, and `run_long_sequence_gst_base`, the "base" variant that fiducial pair reduction relied on for accepting an explicit circuit-structure list. The Protocol API takes an explicit experiment design either way, so there's only one code path to learn.

In [9]:
#use a depolarized version of the target gates to generate the data
mdl_datagen = target_model.depolarize(op_noise=0.1, spam_noise=0.001)

#GateSetTomographyDesign is built around a processor spec, which any
#model can produce.
pspec = target_model.create_processor_spec()

def runGST(gstStructs, exptList):
    #Use list of experiments, expList, to generate some data
    ds = pygsti.data.simulate_data(mdl_datagen, exptList,
            num_samples=1000,sample_error="binomial", seed=1234)

    #Wrap the already-built circuit structures in an experiment design,
    #pair it with the data, and run GateSetTomography on the pair.
    design = pygsti.protocols.GateSetTomographyDesign(pspec, gstStructs)
    data = pygsti.protocols.ProtocolData(design, ds)
    return pygsti.protocols.GateSetTomography(target_model, verbosity=1).run(data)

print("\n------ GST with standard (full) sequences ------")
full_results = runGST(fullStructs, fullExperiments)

print("\n------ GST with PFPR sequences (greedy heuristic) ------")
pfpr_results_greedy = runGST(pfprStructs_greedy, pfprExperiments_greedy)


------ GST with standard (full) sequences ------
--- Iterative GST: [##################################################] 100.0%  568 circuits ---

------ GST with PFPR sequences (greedy heuristic) ------
--- Iterative GST: [##################################################] 100.0%  126 circuits ---


Finally, one can generate reports using GST with reduced-sequences:

In [10]:
pygsti.report.construct_standard_report(full_results, title="Standard GST Strings Example"
                                       ).write_html("../../../tutorial_files/example_stdstrs_report", connected=True)
pygsti.report.construct_standard_report(pfpr_results_greedy, title="Per-germ FPR (Greedy Heuristic) Report Example"
                                        ).write_html("../../../tutorial_files/example_pfpr_greedy_report", connected=True)

Running idle tomography
Computing switchable properties
Found standard clifford compilation from smq1Q_XY


<env>/lib/python3.13/site-packages/plotly/offline/offline.py:150: UserWarning: 
Unrecognized config options supplied: ['showLink', 'linkText']
  warnings.warn(


Running idle tomography
Computing switchable properties
Found standard clifford compilation from smq1Q_XY


<env>/lib/python3.13/site-packages/plotly/offline/offline.py:150: UserWarning: 
Unrecognized config options supplied: ['showLink', 'linkText']
  warnings.warn(


If all has gone well, the <a href="../../../reports/example_stdstrs_report.html">Standard GST</a>
and <a href="../../../reports/example_pfpr_greedy_report.html">PFPR (Greedy)</a>
reports may now be viewed.
The only notable difference between them is the "gaps" in the color box plots, which plot quantities such as the log-likelihood across all operation sequences, organized by germ and fiducials.